In [7]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torchvision.datasets as datasets

from models.mnist.small_cnn_mnist import SmallCNNMNIST
from attacks.pgd import pgd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
])

mnist_test = datasets.MNIST(
    root="../data",
    train=False,
    download=False, 
    transform=transform_mnist
)

mnist_loader = torch.utils.data.DataLoader(
    mnist_test,
    batch_size=128,
    shuffle=False
)

In [4]:
def evaluate(model, loader):
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, pred = outputs.max(1)

            total += y.size(0)
            correct += pred.eq(y).sum().item()

    return 100 * correct / total

In [5]:
model = SmallCNNMNIST().to(device)
model.load_state_dict(torch.load("../models/checkpoints/cnn_mnist.pt", map_location=device))
model.eval()

SmallCNNMNIST(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [8]:
epsilons = [0, 0.01, 0.03, 0.05, 0.1, 0.2]

pgd_results = []

for eps in epsilons:
    correct = 0
    total = 0

    for x, y in mnist_loader:
        x, y = x.to(device), y.to(device)

        if eps == 0:
            x_adv = x
        else:
            x_adv = pgd(
                model, 
                x, 
                y, 
                epsilon=eps,
                alpha=eps/4,     
                num_steps=20,
                random_start=True
            )

        outputs = model(x_adv)
        _, pred = outputs.max(1)

        total += y.size(0)
        correct += pred.eq(y).sum().item()

    acc = 100 * correct / total
    pgd_results.append(acc)

    print(f"PGD | Eps: {eps:.2f} | Acc: {acc:.2f}%")

PGD | Eps: 0.00 | Acc: 98.67%
PGD | Eps: 0.01 | Acc: 98.03%
PGD | Eps: 0.03 | Acc: 95.92%
PGD | Eps: 0.05 | Acc: 92.39%
PGD | Eps: 0.10 | Acc: 66.48%
PGD | Eps: 0.20 | Acc: 1.26%


In [ ]:
classes = test_dataset.classes

data_iter = iter(test_loader)
x, y = next(data_iter)

x, y = x.to(device), y.to(device)

epsilon = 0.03
x_adv = pgd(model, x, y, epsilon,alpha=eps/4, num_steps=20 )


x = x.cpu()
x_adv = x_adv.cpu()

fig, axes = plt.subplots(2, 5, figsize=(12,5))

for i in range(5):
    # original
    axes[0, i].imshow(x[i].permute(1,2,0))
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    # adversarial
    axes[1, i].imshow(x_adv[i].permute(1,2,0))
    axes[1, i].set_title("Adversarial")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()